# 05 - Double DQN Training Loop (Module-First)

Notebook nay goi truc tiep notebook API trong `src.rl.training.notebook_api`.

Ho tro 2 mode:
- pure: train RL tu dau
- warmstart: nap baseline checkpoint + preprocessing artifacts

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd] + list(cwd.parents) if (p / 'src').exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


from src.ml.artifacts import get_ml_checkpoint_path, get_ml_preprocessing_path
from src.rl.training.notebook_api import run_rl
from src.rl.training.runner import RLTrainingConfig

In [2]:
ROOT = Path('/workspace/ai-core')
MODE = os.getenv('RL_MODE', 'warmstart').strip().lower()  # pure | warmstart
RUN_ID = os.getenv('RL_RUN_ID', f'notebook_{MODE}_h15')

CORRIDOR_IDS = [
    14146616491042222, 73904187376705400, 132965186560956307, 136550177913819656,
    392537437542429252, 418854844871232114, 463921826071989712, 499090817621594113,
    553923893084418928, 597258498146003683, 646713380690000556, 647577676530405923,
    665064665204826106, 757793456805938866, 790131348468848924, 807745493105409767,
    870230208041189948, 934115805333902094, 988709510142577156, 1100735735503891924
]

DEFAULT_DATA_PATH = str(ROOT / 'data' / 'processed' / '02_balanced_training_data.parquet')
DATA_PATH = os.getenv('RL_DATA_PATH', DEFAULT_DATA_PATH)

config = RLTrainingConfig(
    run_id=RUN_ID,
    corridor_ids=CORRIDOR_IDS,
    start_date='2026-03-25',
    end_date='2026-04-26',
    peak_hours_only=True,
    
    # Huấn luyện sâu
    episodes=100,
    max_steps_per_episode=10000,
    batch_size=256,              # Có thể nâng lên 256 nếu GPU dư RAM
    learning_rate=0.00005,
    
    # Chiến thuật Exploration
    epsilon_start=1.0,
    epsilon_decay=0.96,          # Chạm đáy 0.05 ở khoảng episode 75         # 
    epsilon_min=0.05,
    
    # Bộ nhớ & Ổn định
    replay_capacity=600000,      # Chứa gần trọn tập train (~611k windows)
    warmup_steps=10000,          # Thu thập kinh nghiệm trong 1 episode đầu
    target_update=3,             # Cập nhật mạng Target sau mỗi 3 episode
    
    # Xử lý mất cân bằng (QUAN TRỌNG)
    use_window_balancing=True,   # Kích hoạt lấy mẫu 50/50 (Kẹt xe / Thoáng)
    use_class_aware_reward=False, # Dùng trọng số phần thưởng cho class hiếm
    
    # Cấu hình dừng sớm
    # early_stop_patience=20,
    # early_stop_warmup_episodes=50,
    
    # Thiết bị & Dữ liệu
    requested_device='cuda',
    data_path=DATA_PATH,
    pretrained_model_path=str(get_ml_checkpoint_path(run_id='manual_h15')),
    artifacts_path=str(get_ml_preprocessing_path(run_id='manual_h15')),
)


print('MODE  :', MODE)
print('RUN_ID:', RUN_ID)
print('DATA  :', config.data_path)
print('Warmstart artifacts:', config.artifacts_path)
print('Warmstart checkpoint:', config.pretrained_model_path)

MODE  : warmstart
RUN_ID: notebook_warmstart_h15
DATA  : /workspace/ai-core/data/processed/02_balanced_training_data.parquet
Warmstart artifacts: /app/artifacts/ml/preprocessing/preprocessing_artifacts_manual_h15.pkl
Warmstart checkpoint: /app/artifacts/ml/checkpoints/best_traffic_model_manual_h15.pt


In [3]:
RUN_TRAIN = True

if MODE not in {'pure', 'warmstart'}:
    raise ValueError("MODE must be 'pure' or 'warmstart'")

if RUN_TRAIN:
    result = run_rl(mode=MODE, config=config)
    print('Training done for mode:', result['mode'])
    print('Run ID:', result['run_id'])
    print('Checkpoint:', result['checkpoint_path'])
    print('History   :', result['history_path'])
    print('Metrics   :', result['metrics_path'])
    if result.get('metrics'):
        final_summary = result['metrics'].get('final_summary', {})
        print('Final summary keys:', sorted(final_summary.keys()))
else:
    print('Dry-run: set RUN_TRAIN=True to execute RL training via run_rl.')

--- RL TRAINING MODE: WARMSTART ---
📦 Config | corridors=[14146616491042222, 73904187376705400, 132965186560956307, 136550177913819656, 392537437542429252, 418854844871232114, 463921826071989712, 499090817621594113, 553923893084418928, 597258498146003683, 646713380690000556, 647577676530405923, 665064665204826106, 757793456805938866, 790131348468848924, 807745493105409767, 870230208041189948, 934115805333902094, 988709510142577156, 1100735735503891924] | start=2026-03-25 | end=2026-04-26 | peak_hours_only=True | episodes=100 | batch_size=256 | eval_ratio=0.2 | seed=42 | max_segments=0 | max_steps=10000 | horizon=15m | target_offset_steps=1 | checkpoint=/app/artifacts/rl/checkpoints/best_rl_agent_warmstart_notebook_warmstart_h15.pt

🚀 CẤU HÌNH FEATURES SỬ DỤNG:
🔹 DYNAMIC    : ['speed_ratio', 'speed_ratio_delta', 'traffic_index', 'delay_seconds']
🔹 STATIC     : ['time_sin', 'time_cos', 'is_peak_hour', 'is_weekend']
🔹 CATEGORICAL: ['tomtom_frc', 'weather_key', 'day_of_week']

⏳ Đang kéo d

2026-05-16 18:46:17,530 - INFO - 🎬 Exporting evaluation predictions for detailed analysis...
2026-05-16 18:46:24,495 - INFO - 📊 Predictions exported to: /app/artifacts/rl/evaluation/predictions_warmstart_notebook_warmstart_h15.parquet


✅ Eval | acc=0.5645 | macro_f1=0.6576
📊 Đã lưu RL metrics vào: /app/artifacts/rl/metrics/rl_metrics_warmstart_notebook_warmstart_h15.json
Training done for mode: warmstart
Run ID: notebook_warmstart_h15
Checkpoint: /app/artifacts/rl/checkpoints/best_rl_agent_warmstart_notebook_warmstart_h15.pt
History   : /app/artifacts/rl/histories/rl_history_warmstart_notebook_warmstart_h15.pkl
Metrics   : /app/artifacts/rl/metrics/rl_metrics_warmstart_notebook_warmstart_h15.json
Final summary keys: ['best_eval_macro_f1', 'best_reward', 'confusion_matrix', 'early_stop_no_improve_count', 'mean_q_value', 'mean_target_q_value', 'mean_td_error', 'num_episodes', 'per_class_metrics', 'stopped_early']


In [4]:
# Optional: inspect RL artifact folders
rl_root = ROOT / 'artifacts' / 'rl'
for subdir in ['checkpoints', 'history', 'metrics']:
    path = rl_root / subdir
    names = [p.name for p in sorted(path.glob('*'))] if path.exists() else []
    print(f'{subdir}:', names[:10])

checkpoints: ['best_rl_agent.pt', 'best_rl_agent_pure.pt', 'best_rl_agent_pure_pure_balanced_gpu.pt', 'best_rl_agent_pure_pure_fast_gpu.pt', 'best_rl_agent_pure_pure_full.pt', 'best_rl_agent_pure_pure_full_gpu.pt', 'best_rl_agent_pure_seed42.pt', 'best_rl_agent_pure_seed43.pt', 'best_rl_agent_pure_seed44.pt', 'best_rl_agent_warmstart.pt']
history: []
metrics: ['rl_metrics_pure.json', 'rl_metrics_pure_pure_balanced_gpu.json', 'rl_metrics_pure_pure_fast_gpu.json', 'rl_metrics_pure_pure_full.json', 'rl_metrics_pure_pure_full_gpu.json', 'rl_metrics_pure_seed42.json', 'rl_metrics_pure_seed43.json', 'rl_metrics_pure_seed44.json', 'rl_metrics_warmstart.json', 'rl_metrics_warmstart_notebook_warmstart_h15.json']
